# v30 A — analysis-only error-family report

Classify wrong YNU cases by quantifier, implication, necessary/sufficient, negation, and unclassified families.

**Current best baseline:** v27 standard, macro-F1 = 0.5833.  
**Safety rule:** If v30 does not safely improve macro-F1, rollback to v27 standard.


In [ ]:

# ================================================================
# Common utilities for EXACT 2026 v30 notebooks
# Offline, no model load, no vLLM. Designed for Kaggle or local run.
# ================================================================
import os, re, json, math, glob, shutil, statistics
from pathlib import Path
from collections import Counter, defaultdict

LABELS = ['A','B','C','D','Yes','No','Unknown']
YNU = {'Yes','No','Unknown'}
MC = {'A','B','C','D'}
BASELINE_V27 = {'acc':0.6154, 'macro_f1':0.5833, 'weighted_f1':0.6110, 'mc_macro':0.5515, 'ynu_macro':0.6258}

SEARCH_DIRS = [
    Path('/kaggle/working'),
    Path('/kaggle/input/datasets/yixuanisthebest/v2333333'),
    Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline'),
    Path('/mnt/data'),
    Path('.'),
]

def find_file(name, required=True):
    candidates = []
    for d in SEARCH_DIRS:
        p = d / name
        if p.exists():
            return p
        candidates.extend(d.glob(f'**/{name}') if d.exists() else [])
    if candidates:
        return candidates[0]
    if required:
        raise FileNotFoundError(f'Missing required file: {name}. Searched: {[str(x) for x in SEARCH_DIRS]}')
    return None

def load_json_file(name_or_path, required=True):
    p = Path(name_or_path)
    if not p.exists():
        p = find_file(str(name_or_path), required=required)
    if p is None:
        return None
    with open(p, 'r', encoding='utf-8') as f:
        obj = json.load(f)
    print(f'Loaded {p} | type={type(obj).__name__}')
    return obj

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print('Saved', path)

def normalize_answer(x):
    if x is None:
        return None
    s = str(x).strip()
    low = s.lower()
    mapping = {
        'yes':'Yes','true':'Yes','y':'Yes',
        'no':'No','false':'No','n':'No',
        'unknown':'Unknown','unk':'Unknown','uncertain':'Unknown','not enough information':'Unknown',
        'a':'A','b':'B','c':'C','d':'D'
    }
    if s in LABELS: return s
    if low in mapping: return mapping[low]
    m = re.search(r'\b(A|B|C|D|Yes|No|Unknown)\b', s, flags=re.I)
    if m:
        return normalize_answer(m.group(1))
    return s

def get_first(d, keys, default=None):
    if not isinstance(d, dict): return default
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default

def flatten_pred_records(obj):
    """Return list of prediction records from common JSON layouts."""
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for key in ['predictions','preds','records','items','data','results','by_sample']:
            val = obj.get(key)
            if isinstance(val, list):
                return val
        # dict keyed by id -> record
        vals = list(obj.values())
        if vals and all(isinstance(v, dict) for v in vals):
            out = []
            for k, v in obj.items():
                vv = dict(v)
                vv.setdefault('id', k)
                out.append(vv)
            return out
    raise ValueError('Cannot infer prediction records layout')

def record_id(rec, idx):
    return get_first(rec, ['id','idx','index','sample_id','question_id','qid','record_id'], idx)

def gold_of(rec):
    return normalize_answer(get_first(rec, ['gold','label','answer','target','gold_answer','true_answer','gt']))

def pred_of(rec):
    return normalize_answer(get_first(rec, ['pred','prediction','selected','selected_answer','final_answer','answer_pred','output']))

def text_of(rec):
    parts=[]
    for k in ['question','query','premises','premise','context','support','explanation','rationale','cot','selected_support','text']:
        v = rec.get(k) if isinstance(rec, dict) else None
        if v is None: continue
        if isinstance(v, (list,tuple)):
            v = ' '.join(map(str, v))
        parts.append(str(v))
    return '\n'.join(parts)

def safe_div(a,b):
    return a/b if b else 0.0

def classification_report_simple(y_true, y_pred, labels=LABELS):
    per = {}
    for lab in labels:
        tp = sum(1 for t,p in zip(y_true,y_pred) if t==lab and p==lab)
        fp = sum(1 for t,p in zip(y_true,y_pred) if t!=lab and p==lab)
        fn = sum(1 for t,p in zip(y_true,y_pred) if t==lab and p!=lab)
        prec = safe_div(tp, tp+fp)
        rec = safe_div(tp, tp+fn)
        f1 = safe_div(2*prec*rec, prec+rec)
        per[lab] = {'precision':prec, 'recall':rec, 'f1':f1, 'support':sum(1 for t in y_true if t==lab), 'pred_count':sum(1 for p in y_pred if p==lab)}
    acc = safe_div(sum(t==p for t,p in zip(y_true,y_pred)), len(y_true))
    macro = sum(per[l]['f1'] for l in labels)/len(labels)
    total = len(y_true)
    weighted = sum(per[l]['f1']*per[l]['support'] for l in labels)/total if total else 0
    mc_labels = ['A','B','C','D']
    ynu_labels = ['Yes','No','Unknown']
    mc_macro = sum(per[l]['f1'] for l in mc_labels)/len(mc_labels)
    ynu_macro = sum(per[l]['f1'] for l in ynu_labels)/len(ynu_labels)
    return {'n':len(y_true), 'acc':acc, 'macro_f1':macro, 'weighted_f1':weighted, 'mc_macro':mc_macro, 'ynu_macro':ynu_macro, 'per_label':per,
            'gold_count':dict(Counter(y_true)), 'pred_count':dict(Counter(y_pred))}

def print_metrics(name, metrics):
    print('\n' + '='*72)
    print(name)
    for k in ['n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro']:
        print(f'{k:14s}: {metrics[k]:.4f}' if isinstance(metrics[k], float) else f'{k:14s}: {metrics[k]}')
    print('per-label:')
    for lab, d in metrics['per_label'].items():
        print(f"  {lab:7s} f1={d['f1']:.4f} p={d['precision']:.4f} r={d['recall']:.4f} support={d['support']} pred={d['pred_count']}")

def load_base_v27():
    obj = load_json_file('v27_standard_preds.json')
    records = flatten_pred_records(obj)
    rows = []
    for i, rec in enumerate(records):
        rid = record_id(rec, i)
        gold = gold_of(rec)
        pred = pred_of(rec)
        if gold is None or pred is None:
            raise ValueError(f'Record {i} missing gold/pred. keys={list(rec.keys())}')
        rows.append({'i':i, 'id':rid, 'gold':gold, 'pred':pred, 'raw':rec, 'text':text_of(rec)})
    return rows

def maybe_load_candidates():
    p = find_file('v23_val_candidates.json', required=False)
    if p is None:
        print('Optional candidates not found: v23_val_candidates.json')
        return {}
    obj = load_json_file(p)
    # Return dict id/index -> list of candidate records/texts
    out = defaultdict(list)
    if isinstance(obj, list):
        iterable = obj
    elif isinstance(obj, dict):
        iterable = obj.get('records') or obj.get('data') or obj.get('candidates') or obj.get('items')
        if iterable is None:
            # keyed by id layout
            for k,v in obj.items():
                if isinstance(v, list): out[k].extend(v)
                elif isinstance(v, dict): out[k].append(v)
            return out
    else:
        return out
    for i, rec in enumerate(iterable):
        if not isinstance(rec, dict):
            out[i].append(rec); continue
        rid = record_id(rec, i)
        cands = get_first(rec, ['candidates','samples','outputs','generations','answers'], None)
        if isinstance(cands, list): out[rid].extend(cands)
        else: out[rid].append(rec)
    return out

def candidate_answers_and_text(cands):
    ans = []
    texts = []
    for c in cands or []:
        if isinstance(c, dict):
            a = normalize_answer(get_first(c, ['answer','pred','prediction','final_answer','selected_answer','label'], None))
            t = text_of(c)
            if not t:
                t = json.dumps(c, ensure_ascii=False)[:2000]
        else:
            t = str(c)
            a = normalize_answer(t)
        if a in YNU or a in MC:
            ans.append(a)
        texts.append(t)
    return ans, '\n'.join(texts)

# ----------------------------- targeted detectors -----------------------------
QUANT_ALL = r'\b(all|every|each|any|everyone|everybody|always|tất cả|mọi|mỗi|bất kỳ|luôn)\b'
QUANT_SOME = r'\b(some|at least one|exists|exist|there is|there are|một số|có ít nhất|tồn tại|có vài)\b'
IMPL_WORDS = r'\b(if|then|only if|implies|whenever|provided that|sufficient|necessary|requires|required|prerequisite|nếu|thì|chỉ nếu|kéo theo|điều kiện cần|điều kiện đủ|yêu cầu)\b'
OVERCLAIM_WORDS = r'\b(prove|guarantee|ensure|must|definitely|always|necessarily|chắc chắn|đảm bảo|bắt buộc|phải|luôn|tất yếu)\b'
NEGATION_WORDS = r'\b(not|no|never|cannot|neither|không|chưa|không thể|không phải)\b'
CONTRA_WORDS = r'\b(however|but|although|contradict|contrary|inconsistent|except|nhưng|tuy nhiên|mâu thuẫn|ngoại trừ|trái lại)\b'

def low(s): return (s or '').lower()

def flag_quantifier_overclaim(question_text, evidence_text):
    q = low(question_text); e = low(evidence_text)
    if re.search(QUANT_ALL, q) and re.search(QUANT_SOME, e) and not re.search(QUANT_ALL, e):
        return True
    if re.search(r'\b(exists|there is|at least one|tồn tại|có ít nhất)\b', q) and re.search(r'\b(all|every|none|no one|tất cả|không ai)\b', e):
        # suspicious mismatch, not always a flip by itself
        return True
    return False

def flag_implication_trap(question_text, evidence_text):
    q = low(question_text); e = low(evidence_text)
    # Conservative: require implication language in question and mismatch/trap keywords in evidence.
    if re.search(IMPL_WORDS, q) and re.search(IMPL_WORDS, e):
        if re.search(r'\b(converse|inverse|necessary|sufficient|only if|unless|đảo|ngược|cần|đủ|chỉ nếu)\b', q + ' ' + e):
            return True
    return False

def flag_overclaim_text(question_text, evidence_text):
    q = low(question_text); e = low(evidence_text)
    return bool(re.search(OVERCLAIM_WORDS, q) and (re.search(CONTRA_WORDS, e) or re.search(NEGATION_WORDS, e)))

def categorize_family(question_text, evidence_text):
    fam = []
    if flag_quantifier_overclaim(question_text, evidence_text): fam.append('quantifier_overclaim')
    if flag_implication_trap(question_text, evidence_text): fam.append('implication_converse_necessary_sufficient')
    if flag_overclaim_text(question_text, evidence_text): fam.append('overclaim_with_contra_or_negation')
    if re.search(NEGATION_WORDS, low(question_text) + ' ' + low(evidence_text)): fam.append('negation_present')
    if not fam: fam.append('unclassified')
    return fam

def should_flip_yes_to_no(row, candidates_by_id):
    """Very conservative v30 standard gate: only Yes->No, only YNU, only with >=2 signals."""
    if row['pred'] != 'Yes' or row['gold'] not in YNU:
        return False, []
    raw = row['raw']
    qtext = str(get_first(raw, ['question','query','statement','prompt'], ''))
    base_text = row['text']
    cands = candidates_by_id.get(row['id']) or candidates_by_id.get(row['i']) or []
    cand_ans, cand_text = candidate_answers_and_text(cands)
    evidence = '\n'.join([base_text, cand_text])
    reasons = []
    if flag_quantifier_overclaim(qtext, evidence): reasons.append('quantifier_overclaim')
    if flag_implication_trap(qtext, evidence): reasons.append('implication_trap')
    if flag_overclaim_text(qtext, evidence): reasons.append('overclaim_contra_negation')
    # extra safety: require No candidate OR explicit contradiction/negation in explanation/support
    has_no_candidate = ('No' in cand_ans)
    has_contra = bool(re.search(CONTRA_WORDS, low(evidence)))
    has_neg = bool(re.search(NEGATION_WORDS, low(evidence)))
    if has_no_candidate: reasons.append('has_no_candidate')
    if has_contra: reasons.append('has_contra_text')
    if has_neg: reasons.append('has_negation_text')
    # Conservative gate: at least one structural reason and at least one support reason.
    structural = {'quantifier_overclaim','implication_trap','overclaim_contra_negation'} & set(reasons)
    support = {'has_no_candidate','has_contra_text','has_negation_text'} & set(reasons)
    return bool(structural and support), reasons

def make_summary(version, base_metrics, new_metrics, n_flips=0, flips=None, extra=None):
    extra = extra or {}
    delta_macro = new_metrics['macro_f1'] - base_metrics['macro_f1']
    selected = (new_metrics['macro_f1'] > base_metrics['macro_f1'] and new_metrics['macro_f1'] >= BASELINE_V27['macro_f1'] and n_flips > 0)
    # block obvious label collapse
    for lab, d in new_metrics['per_label'].items():
        if d['support'] > 0 and d['pred_count'] == 0:
            selected = False
            extra.setdefault('warnings', []).append(f'COLLAPSE_{lab}')
    return {
        'version': version,
        'baseline_v27_expected': BASELINE_V27,
        'base_metrics': base_metrics,
        'new_metrics': new_metrics,
        'delta_macro_f1': delta_macro,
        'n_flips': n_flips,
        'flips_preview': (flips or [])[:30],
        'selection_status': 'SELECT_V30' if selected else 'ROLLBACK_TO_V27',
        **extra,
    }

# ================================================================
# v30 A: analysis-only YNU error-family report
# Does not produce final predictions. Purpose: classify wrong YNU cases.
# ================================================================
rows = load_base_v27()
candidates_by_id = maybe_load_candidates()
y_true = [r['gold'] for r in rows]
y_base = [r['pred'] for r in rows]
base_metrics = classification_report_simple(y_true, y_base)
print_metrics('BASE v27 standard', base_metrics)

error_rows = []
for row in rows:
    if row['gold'] not in YNU: continue
    if row['gold'] == row['pred']: continue
    raw = row['raw']
    qtext = str(get_first(raw, ['question','query','statement','prompt'], ''))
    cands = candidates_by_id.get(row['id']) or candidates_by_id.get(row['i']) or []
    cand_ans, cand_text = candidate_answers_and_text(cands)
    evidence = row['text'] + '\n' + cand_text
    fam = categorize_family(qtext, evidence)
    error_rows.append({
        'i': row['i'], 'id': row['id'], 'gold': row['gold'], 'pred': row['pred'],
        'families': fam, 'candidate_answer_counts': dict(Counter(cand_ans)),
        'question': qtext[:500], 'text_preview': evidence[:1000]
    })

family_counts = Counter(f for r in error_rows for f in r['families'])
by_confusion = Counter((r['gold'], r['pred']) for r in error_rows)
report = {
    'version': 'v30_a_error_family_report',
    'purpose': 'Analysis only; categorize wrong YNU cases. Do not use as final prediction.',
    'base_metrics': base_metrics,
    'n_wrong_ynu': len(error_rows),
    'family_counts': dict(family_counts),
    'confusion_counts': {f'{a}->{b}': c for (a,b),c in by_confusion.items()},
    'wrong_cases': error_rows,
}
save_json(report, '/kaggle/working/v30_a_summary.json')
save_json(error_rows, '/kaggle/working/v30_a_preds.json')
save_json(report, 'v30_a_summary.json')
save_json(error_rows, 'v30_a_preds.json')
print('Family counts:', family_counts)
print('Confusion counts:', by_confusion)
report
